# Fleet Maintenance Planning

**Team:** 38

### 1. Project Overview

#### Objective

Predict which vehicles should be inspected and determine
the appropriate spare-parts budget for each inspected vehicle.

The objective is to minimise the total maintenance cost
using the competition's specified cost function.

#### Project Workflow

1. Data loading and exploration
2. Preprocessing and feature selection
3. Failure prediction
4. Repair-cost prediction
5. Cost-based decision optimisation
6. Final submission generation

### 2. Imports and Config

In [3]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42

# Find project root by locating the data folder
current = Path.cwd().resolve()

ROOT = next(
    (p for p in [current, *current.parents]
     if (p / "data").is_dir() and (p / "src").is_dir()),
    None
)

if ROOT is None:
    raise FileNotFoundError(
        "Could not locate project root containing data/ and src/"
    )

os.chdir(ROOT)

# Project directories
DATA_DIR = ROOT / "data"
SRC_DIR = ROOT / "src"
RESULTS_DIR = ROOT / "results"
SUBMISSIONS_DIR = ROOT / "submissions"

print("Project root:", ROOT)
print("Working directory:", Path.cwd())
print("Data directory:", DATA_DIR)

Project root: C:\Users\Avnish\Documents\GitHub\fleet_maintenance_p2_team38
Working directory: C:\Users\Avnish\Documents\GitHub\fleet_maintenance_p2_team38
Data directory: C:\Users\Avnish\Documents\GitHub\fleet_maintenance_p2_team38\data


### 3. Data Loading:

In [4]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

sample_submission = pd.read_csv(
    DATA_DIR / "sample_submission.csv"
)

print("Training data:", train_df.shape)
print("Testing data:", test_df.shape)
print("Sample submission:", sample_submission.shape)

Training data: (56000, 174)
Testing data: (20000, 171)
Sample submission: (20000, 3)


### 4. Data Validation

In [5]:
# Check required columns
assert "failure" in train_df.columns
assert "repair_cost" in train_df.columns
assert "id" in train_df.columns

# Check identifiers
assert train_df["id"].is_unique
assert test_df["id"].is_unique

# Check target
assert train_df["failure"].notna().all()
assert train_df["failure"].isin([0, 1]).all()

# Check feature compatibility
sensor_cols = [
    col for col in train_df.columns
    if col.startswith("s")
]

assert set(sensor_cols) == {
    col for col in test_df.columns
    if col.startswith("s")
}

print("Basic data validation passed!")

Basic data validation passed!


### 5. EDA Summary

- The failure rate is highly imbalanced (0.71%).
- Repair costs are strongly right-skewed.
- Sensor features contain substantial missing values.
- Eight sensors have more than 50% missing values.
- One sensor (`s146`) is constant.
- Several sensors show substantial train–test differences, suggesting potential distribution shift between depots.

### 6. Cross-Validation Setup

In [7]:
from src.cv_utils import prepare_training_data, load_folds, validate_folds

# Prepare features and target
X, y = prepare_training_data(train_df)

# Load shared folds
folds = load_folds()

# Validate
validate_folds(folds, len(X))

print(f"Loaded {len(folds)} CV folds.")

Fold validation successful.
Loaded 5 CV folds.
